In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

import pandas as pd
import numpy as np
from forge_class_instance3 import *
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr, spearmanr
from lifelines.utils import concordance_index
from sklearn.metrics import r2_score
%matplotlib inline

In [ ]:
forge_model = FORGE.load_forge(path='/home/sreeramp/cancer_dependency_project/nilabja/Approach3_Latent_factor/git_repo/Models/optuna_models/ERLOTINIB_EGFR_forgeModel_optuna100.pkl')
# forge_model.best_hyperparams

In [ ]:
exp_data = pd.read_csv(forge_model.exp_path, header = 0, index_col = 0)
dep_data = pd.read_csv(forge_model.dep_path, header = 0, index_col = 0)
ic50_data = pd.read_csv(forge_model.ic50_path, header = 0, index_col = 0)
ic50_data = forge_model.ic50_data.T
exp_data.shape, dep_data.shape, ic50_data.shape

In [ ]:
common_cellLines = list(set(exp_data.index) & set(dep_data.index) & set(ic50_data.index))
len(common_cellLines)

In [ ]:
exp_data_subset = exp_data.loc[common_cellLines, forge_model.hcg_list].drop_duplicates(keep='first')
dep_target = dep_data.loc[common_cellLines, forge_model.target].dropna().drop_duplicates(keep='first')
drug_ic50 = ic50_data.loc[common_cellLines, forge_model.drug].dropna().drop_duplicates(keep='first')
exp_data_subset = exp_data.loc[dep_target.index, forge_model.hcg_list]
drug_ic50 = ic50_data.loc[dep_target.index, forge_model.drug]
exp_data_subset.shape, dep_target.shape, drug_ic50.shape

In [ ]:
exp_data_subset.head()

In [ ]:
dep_target.head()

In [ ]:
drug_ic50.head()

In [ ]:
forge_model.exp_data = exp_data_subset
forge_model.dep_data = dep_target
forge_model.ic50_data = drug_ic50
forge_model.exp_data.shape, forge_model.dep_data.shape, forge_model.ic50_data.shape

In [ ]:
common_train_cellLines = list(set(forge_model.train_cellLines) & set(forge_model.exp_data.index))
common_test_cellLines = list(set(forge_model.test_cellLines) & set(forge_model.exp_data.index))

In [ ]:
all_cellLines = common_train_cellLines[:]
all_cellLines.extend(common_test_cellLines)
len(all_cellLines)

In [ ]:
G_train = forge_model.exp_data.loc[common_train_cellLines, :].to_numpy()
G_test  = forge_model.exp_data.loc[common_test_cellLines, :].to_numpy()

D_train = forge_model.dep_data.loc[common_train_cellLines].to_numpy().reshape(-1, 1)
D_test  = forge_model.dep_data.loc[common_test_cellLines].to_numpy().reshape(-1, 1)

I_train = forge_model.ic50_data.loc[common_train_cellLines].to_numpy().reshape(-1, 1)
I_test  = forge_model.ic50_data.loc[common_test_cellLines].to_numpy().reshape(-1, 1)

G_train.shape, D_train.shape, I_train.shape, G_test.shape, D_test.shape, I_test.shape

In [ ]:
## Construct weight matrices
dep_imp = forge_model.W @ forge_model.hD
ic50_imp = forge_model.W @ forge_model.hI
gene_imp_df = pd.DataFrame({'dep_imp': dep_imp.flatten(),
                            'ic50_imp': ic50_imp.flatten()},
                           index = forge_model.hcg_list)
gene_imp_df.head()

In [ ]:
gene_imp_df['combined'] = gene_imp_df['dep_imp'] - gene_imp_df['ic50_imp']
gene_imp_df['scaled_combined'] = (gene_imp_df['combined'] - np.mean(gene_imp_df['combined'])) / np.std(gene_imp_df['combined'])
gene_imp_df.head()

In [ ]:
gene_imp_df.to_csv('/home/sreeramp/cancer_dependency_project/forge_v2/FORGE/intermediate_data/EGFR_ERLOTINIB_geneImp_optuna.csv', index=True)

#### Gene importance plot

In [ ]:
gene_imp_df['gene'] = gene_imp_df.index
top_5 = gene_imp_df.sort_values(by='scaled_combined', ascending=False).head(5)['gene'].tolist()
bottom_5 = gene_imp_df.sort_values(by='scaled_combined', ascending=True).head(5)['gene'].tolist()
gene_imp_df['marker'] = np.where(gene_imp_df['gene'].isin(top_5), 'top_5', np.where(gene_imp_df['gene'].isin(bottom_5), 'bottom_5', 'normal'))
gene_imp_df.head()

In [ ]:
gene_imp_df['scaled_combined'].describe()

In [ ]:
custom_palette = {'top_5': 'green', 'bottom_5': 'blue', 'normal': 'grey'}
# --------------------------------------------------
# Create a plotting copy and flip IC50 importance
# --------------------------------------------------
plot_df = gene_imp_df.copy()
plot_df['ic50_imp_neg'] = -1 * plot_df['ic50_imp']

# --------------------------------------------------
# Add jitter (small noise)
# --------------------------------------------------
jitter_strength = 0.01  # tune if needed
plot_df['dep_jit'] = plot_df['dep_imp'] + np.random.normal(0, jitter_strength, size=len(plot_df))
plot_df['ic50_jit'] = plot_df['ic50_imp_neg'] + np.random.normal(0, jitter_strength, size=len(plot_df))

# --------------------------------------------------
# Scatter plot
# --------------------------------------------------
plt.figure(figsize=(8, 7))

sns.scatterplot(
    x='dep_jit',
    y='ic50_jit',
    hue='marker',
    palette=custom_palette,
    data=plot_df,
    alpha=0.7,
    s=40
)

# --------------------------------------------------
# Zero reference lines
# --------------------------------------------------
plt.axhline(0, linestyle='--', color='grey', linewidth=1)
plt.axvline(0, linestyle='--', color='grey', linewidth=1)

# --------------------------------------------------
# Label only top_20 and bottom_20
# --------------------------------------------------
label_df = plot_df[plot_df['marker'].isin(['top_5', 'bottom_5'])]

for _, row in label_df.iterrows():
    plt.text(
        row['dep_jit'],
        row['ic50_jit'],
        row['gene'],          # change if your label column has a different name
        fontsize=8,
        alpha=0.9
    )

# --------------------------------------------------
# Cosmetics
# --------------------------------------------------
plt.xlabel('Dependency importance')
plt.ylabel('-IC50 importance')
plt.title('Gene importance: Dependency vs IC50')
plt.legend(title='Marker', frameon=False)
sns.despine()

plt.tight_layout()
# plt.savefig('/home/sreeramp/cancer_dependency_project/nilabja/Approach3_Latent_factor/git_repo/Figs/egfr_erlotinib_geneImp_top10_optuna.pdf', dpi=300)
plt.show()


#### Benefit score computations

In [ ]:
full_exp_data = forge_model.exp_data.loc[all_cellLines, gene_imp_df.index]
full_exp_data_scaled = (full_exp_data - forge_model.mean_exp) / forge_model.std_exp
D_train, D_test = D_train - forge_model.mean_dep, D_test - forge_model.mean_dep
I_train, I_test = I_train - forge_model.mean_ic50, I_test - forge_model.mean_ic50
full_exp_data_scaled.shape, gene_imp_df.shape, D_train.shape, I_train.shape

In [ ]:
Z_matrix = full_exp_data_scaled.to_numpy() @ forge_model.W
Z_matrix.shape

In [ ]:
z_values_df = pd.DataFrame(Z_matrix, index=all_cellLines, columns=[f'dim{i}' for i in range(Z_matrix.shape[1])])
z_values_df['scaled_dep'] = D_train.flatten().tolist() + D_test.flatten().tolist()
z_values_df['scaled_ic50'] = I_train.flatten().tolist() + I_test.flatten().tolist()
z_values_df['pred_dep'] = (z_values_df[[f'dim{i}' for i in range(Z_matrix.shape[1])]].to_numpy() @ forge_model.hD).flatten() 
z_values_df['pred_ic50'] = (z_values_df[[f'dim{i}' for i in range(Z_matrix.shape[1])]].to_numpy() @ forge_model.hI).flatten()
z_values_df.head()

In [ ]:
z_values_df['set'] = ['train' if cl in common_train_cellLines else 'test' for cl in z_values_df.index]
z_values_df['cell_line'] = z_values_df.index
z_values_df['benefit_score_raw'] = z_values_df['pred_dep'] - z_values_df['pred_ic50']
z_values_df['benefit_score_scaled'] = (z_values_df['benefit_score_raw'] - np.min(z_values_df['benefit_score_raw'])) / (np.max(z_values_df['benefit_score_raw']) - np.min(z_values_df['benefit_score_raw']))
z_values_df.head()

In [ ]:
z_values_df['benefit_score_scaled'].describe()

In [ ]:
z_plot_df = z_values_df.drop(columns=[i for i in z_values_df.columns if not i.startswith('dim')])
z_plot_df = z_plot_df.sort_index()
z_values_df = z_values_df.loc[z_plot_df.index]
z_plot_df.head()

In [ ]:
from matplotlib.colors import LinearSegmentedColormap, Normalize
from scipy.spatial.distance import pdist
from scipy.cluster.hierarchy import linkage
from matplotlib import rcParams
from dynamicTreeCut import cutreeHybrid
from matplotlib.patches import Patch

In [ ]:
# === 3. Hierarchical Clustering and Annotation Color Mapping (on Combined Data) ===

print("Performing hierarchical clustering on the combined dataset...")

# --- Step 3a: Compute linkage matrices for the combined data ---
row_dist = pdist(z_plot_df.values, metric='euclidean')
row_link = linkage(row_dist, method='ward')
col_link = linkage(pdist(z_plot_df.T.values), method='ward')

# --- Step 3b: Run Dynamic Tree Cut to find clusters ---
cluster_result = cutreeHybrid(
    row_link, row_dist, deepSplit=4, minClusterSize=13)  # Set a min size
cluster_labels = pd.Series(
    cluster_result["labels"], index=z_plot_df.index, name="Cluster")
num_clusters = len(np.unique(cluster_labels[cluster_labels != 0]))
print(f"Dynamic Tree Cut found {num_clusters} clusters.")

# --- Step 3c: Create annotation colors ---
row_colors_df = pd.DataFrame(index=z_plot_df.index)

# Annotation 1: Dynamic Tree Cut Clusters
unique_clusters = sorted(np.unique(cluster_labels))
palette = sns.color_palette("tab20", len(unique_clusters))
cluster_lut = dict(zip(unique_clusters, palette))
cluster_lut[0] = 'lightgrey'  # Unclustered points
row_colors_df["Cluster"] = cluster_labels.map(cluster_lut)

# Annotation 2: Train vs. Test Set
set_lut = {"train": "#2ca02c", "test": "#ff7f0e"}  # Green and Orange
row_colors_df["set"] = z_values_df["set"].map(set_lut)

# White bar to add space between cluster and set
row_colors_df[" "] = "#ffffff"
# Reorder to force the stacking order: Cluster (top), Spacer, then Set
row_colors_df = row_colors_df[["Cluster", " ", "set"]]


# === 4. Define Colorblind-Friendly Colormaps ===
deep_blue   = '#3B4CC0'   # strong blue
soft_blue   = '#8DB0FE'   # light blue
neutral     = '#F7F7F7'   # keep neutral
soft_orange = '#F4A582'   # light orange
deep_orange = '#B40426'   # strong orange
green_pink_cmap = LinearSegmentedColormap.from_list('green_pink_cmap', [(
    0.0, deep_blue), (0.35, soft_blue), (0.5, neutral), (0.65, soft_orange), (1.0, deep_orange)], N=256)
muted_reds = LinearSegmentedColormap.from_list(
    "muted_reds", ["mistyrose", "indianred", "firebrick"])
benefit_cmap = sns.diverging_palette(
    240, 10, as_cmap=True)  # Blue to Red for benefit score

# === 5. Generate the Final Clustered Heatmap ===
print("Generating the final heatmap...")
cg = sns.clustermap(
    z_values_df,
    row_linkage=row_link,
    col_linkage=col_link,
    row_colors=row_colors_df,  # Pass the DataFrame with both annotations
    cmap=green_pink_cmap,
    figsize=(12, 15),
    cbar_pos=None,  # We will create all colorbars manually
    dendrogram_ratio=(0.04, 0.03),
    linewidths=0
)

# Rasterize the heatmap blocks (all quadmesh artists in heatmap)
for artist in cg.ax_heatmap.collections:
    artist.set_rasterized(True)

# === 6. Fine-tune Layout and Add Side Plots ===
fig = cg.fig
heatmap_ax = cg.ax_heatmap
heatmap_pos = heatmap_ax.get_position()


# --- Adjust annotation and dendrogram positions ---
cg.ax_row_colors.set_position(
    [heatmap_pos.x0 - 0.04, heatmap_pos.y0, 0.03, heatmap_pos.height])
if cg.ax_col_colors is not None:
    cg.ax_col_colors.set_visible(False)


# cg.ax_col_colors.set_visible(False) # Hide column colors if they exist
dendro_ax = cg.ax_row_dendrogram
dendro_ax.set_position([0.02, heatmap_pos.y0, 0.05, heatmap_pos.height])

cg.ax_row_dendrogram.set_visible(False)
cg.ax_col_dendrogram.set_visible(False)


# --- Configure heatmap axes and labels ---
row_order = cg.dendrogram_row.reordered_ind
ordered_labels = z_plot_df.index[row_order]
tick_labels = [label if i % 10 == 0 else "" for i,
               label in enumerate(ordered_labels)]  # Adjust frequency
heatmap_ax.set_yticks(np.arange(len(ordered_labels)) + 0.5)
# heatmap_ax.set_yticklabels(tick_labels, fontsize=6.5, rotation=0)
heatmap_ax.set_yticklabels([])
heatmap_ax.set_yticks([])

heatmap_ax.tick_params(axis='y', labelright=False,
                       labelleft=True, length=0, pad=1)

# --- Add Bar Plot 1: EGFR Dependency ---
min_dep, max_dep = z_values_df['scaled_dep'].min(
), z_values_df['scaled_dep'].max()
egfr_ax = fig.add_axes(
    [heatmap_pos.x1 + 0.015, heatmap_pos.y0, 0.03, heatmap_pos.height])
egfr_vals = z_values_df.loc[ordered_labels, "scaled_dep"]
egfr_colors = muted_reds(Normalize(vmin=0, vmax=1)(egfr_vals))
egfr_ax.barh(np.arange(len(egfr_vals)), egfr_vals, height=1.0,
             color=egfr_colors, edgecolor='black', linewidth=0.1)
egfr_ax.set_xlim(min_dep - 0.1, max_dep + 0.1)
egfr_ax.set_ylim(0, len(ordered_labels))
egfr_ax.set_xticks([min_dep - 0.1, 0, max_dep + 0.1])
egfr_ax.set_xticklabels([
    f"{min_dep - 0.1:.1f}",
    "0.0",
    f"{max_dep + 0.1:.1f}"
], fontsize=8, rotation=90)
egfr_ax.set_yticks([])
egfr_ax.set_title("EGFR\nDependency", fontsize=9, rotation=90)
for bar in egfr_ax.patches:
    bar.set_rasterized(True)

# --- Add Bar Plot 2: IC50 ---
ic50_ax = fig.add_axes(
    [heatmap_pos.x1 + 0.053, heatmap_pos.y0, 0.04, heatmap_pos.height])
min_ic50, max_ic50 = z_values_df['scaled_ic50'].min(
), z_values_df['scaled_ic50'].max()
ic50_vals = z_values_df.loc[ordered_labels, "scaled_ic50"]
ic50_colors = muted_reds(Normalize(vmin=-5, vmax=15)(ic50_vals))
ic50_ax.barh(np.arange(len(ic50_vals)), ic50_vals, height=1.0,
             color=ic50_colors, edgecolor='black', linewidth=0.1)
ic50_ax.set_xlim(min_ic50 - 2, max_ic50 + 2)
ic50_ax.set_ylim(0, len(ordered_labels))
ic50_ax.set_xticks([min_ic50 - 2, 0, max_ic50 + 2])
ic50_ax.set_xticklabels([
    f"{min_ic50 - 2:.1f}",
    "0.0",
    f"{max_ic50 + 2:.1f}"
], fontsize=8, rotation=90)
ic50_ax.set_yticks([])
ic50_ax.set_title("IC50", fontsize=9, loc='center', pad=5)
for bar in ic50_ax.patches:
    bar.set_rasterized(True)


# --- Add Bar Plot 3: Benefit Score ---
benefit_ax = fig.add_axes(
    [heatmap_pos.x1 + 0.1, heatmap_pos.y0, 0.03, heatmap_pos.height])
benefit_vals = z_values_df.loc[ordered_labels, "benefit_score_scaled"]
benefit_colors = muted_reds(
    Normalize(vmin=benefit_vals.min(), vmax=benefit_vals.max())(benefit_vals))
# benefit_norm = Normalize(vmin=benefit_vals.min(), vmax=benefit_vals.max())
# benefit_ax.barh(np.arange(len(benefit_vals)), benefit_vals, height=1.0, color=benefit_cmap(benefit_norm(benefit_vals)))
benefit_ax.barh(np.arange(len(benefit_vals)), benefit_vals, height=1.0,
                color=benefit_colors, edgecolor='black', linewidth=0.1)
benefit_ax.set_xticks([0, 0.5, 1])
benefit_ax.set_xticklabels([
    "0",
    "0.5",
    "1"
], fontsize=8, rotation=90)
benefit_ax.set_ylim(0, len(ordered_labels))
benefit_ax.set_yticks([])
benefit_ax.tick_params(axis='x', labelsize=8)
benefit_ax.set_title("Benefit\nScore", fontsize=9, loc='center', pad=5)
for bar in benefit_ax.patches:
    bar.set_rasterized(True)


# --- Add Manual Colorbar for Heatmap ---
# cbar_ax = fig.add_axes([heatmap_pos.x1 + 0.15, heatmap_pos.y0, 0.015, heatmap_pos.height * 0.4])
cbar_ax = fig.add_axes([
    heatmap_pos.x1 + 0.14,                                 # right of bar plots
    heatmap_pos.y0 + heatmap_pos.height * 0.3,             # vertically centered
    0.019,
    heatmap_pos.height * 0.4
])
vmin, vmax = z_plot_df.values.min(), z_plot_df.values.max()
norm = Normalize(vmin=vmin, vmax=vmax)
sm = plt.cm.ScalarMappable(cmap=green_pink_cmap, norm=norm)
cbar = fig.colorbar(sm, cax=cbar_ax)
cbar.ax.tick_params(labelsize=8)
cbar.set_label('Embedding', fontsize=10, rotation=270, labelpad=15)


# --- Add Manual Legend for Cluster and Set Annotations ---
legend_ax = fig.add_axes([heatmap_pos.x1 + 0.14, heatmap_pos.y0 +
                         heatmap_pos.height*0.80, 0.015, heatmap_pos.height*0.25])
handles = [Patch(facecolor=color, label=label)
           for label, color in set_lut.items()]
legend_ax.legend(handles=handles, title='Set',
                 frameon=False, loc='lower left', fontsize=9)
legend_ax.axis('off')
# --- Add Manual Legend for Cluster and Set Annotations --


# --- Final Titles and Layout Adjustments ---
heatmap_ax.set_xlabel("Latent Feature", fontsize=12)
heatmap_ax.set_ylabel("")
num_cell_lines = len(z_values_df['cell_line'].unique())
fig.suptitle(
    f"Cell Line Latent Embeddings (Train & Test, N={num_cell_lines})", y=0.99, fontsize=16)

# fig.savefig("/home/sreeramp/cancer_dependency_project/nilabja/Approach3_Latent_factor/git_repo/revision_final/EGFR_Erlotinib_heatmap.pdf", dpi=600, bbox_inches='tight')

plt.show()

In [ ]:
# z_values_df.to_csv('/home/sreeramp/cancer_dependency_project/forge_v2/FORGE/intermediate_data/egfr_erlotinib_fullRes_optuna.csv', index=True)

#### Identifying top clusters

In [ ]:
# z_values_df = pd.read_csv('/home/sreeramp/cancer_dependency_project/forge_v2/FORGE/intermediate_data/egfr_erlotinib_fullRes_optuna.csv', index=True)

In [ ]:
# z_values_df = z_values_df.loc[z_plot_df.index]
z_values_df['Cluster'] = pd.Series(cluster_result["labels"], index=z_values_df.index)
print(f"Cell lines assigned to {len(z_values_df['Cluster'].unique()) - 1} clusters.")


# === 4. Identify the Target Cluster Based on Your Criteria ===
# This section programmatically finds the cluster that best matches your description.
#
print("\n--- Identifying Target Cluster (Low IC50, High Dependency, High Benefit) ---")
# Exclude unassigned points (cluster 0)
cluster_df = z_values_df[z_values_df['Cluster'] != 0].copy()

# Calculate the average of the key metrics for each cluster
cluster_metrics = cluster_df.groupby('Cluster').agg(
    Avg_IC50=('scaled_ic50', 'mean'),
    Avg_Dependency=('scaled_dep', 'mean'),
    Avg_Benefit_Score=('benefit_score_scaled', 'mean'),
    Size=('Cluster', 'size')
)

# Rank clusters for each metric. Lower rank is better.
# For IC50, we want the lowest value, so we rank ascending.
cluster_metrics['Rank_IC50'] = cluster_metrics['Avg_IC50'].rank(method='first', ascending=True)
# For Dependency and Benefit, we want the highest values, so we rank descending.
cluster_metrics['Rank_Dependency'] = cluster_metrics['Avg_Dependency'].rank(method='first', ascending=False)
cluster_metrics['Rank_Benefit'] = cluster_metrics['Avg_Benefit_Score'].rank(method='first', ascending=False)

# Calculate a total rank score. The cluster with the minimum sum is the best match.
cluster_metrics['Total_Rank'] = cluster_metrics['Rank_IC50'] + cluster_metrics['Rank_Dependency'] + cluster_metrics['Rank_Benefit']

# Find the cluster with the best (lowest) total rank
target_cluster_id = cluster_metrics['Total_Rank'].idxmin()

# Display the summary for the identified cluster to confirm it's the right one
print("\n--- Best Matching Cluster Found ---")
print(f"Cluster ID: {target_cluster_id}")
print("This cluster has the best combined rank for Low IC50, High Dependency, and High Benefit Score.")
print("\nCluster Statistics:")
print(cluster_metrics.loc[target_cluster_id])

In [ ]:
# === 5. Extract and Display the Cell Line Names ===
# Now, filter the main DataFrame to get the names from the target cluster.
#
print(f"\n--- Extracting Cell Lines from Cluster {target_cluster_id} ---")

target_cell_lines = z_values_df[z_values_df['Cluster'] == target_cluster_id]

# Separate into Train and Test sets for clarity
target_train_cell_lines = target_cell_lines[target_cell_lines['set'] == 'train'].index.tolist()
target_test_cell_lines = target_cell_lines[target_cell_lines['set'] == 'test'].index.tolist()

print(f"\nFound {len(target_cell_lines)} total cell lines in this cluster.")

print(f"\nTRAINING Cell Lines ({len(target_train_cell_lines)}):")
if target_train_cell_lines:
    print(target_train_cell_lines)
else:
    print("None")

print(f"\nTEST Cell Lines ({len(target_test_cell_lines)}):")
if target_test_cell_lines:
    print(target_test_cell_lines)
else:
    print("None")

In [ ]:
best_cellLines = target_test_cell_lines[:]
best_cellLines.extend(target_train_cell_lines)
len(best_cellLines) # 26 

In [ ]:
best_cellLines

In [ ]:
# Assume `plot_df` and `ic50_order` are already defined in your environment
ic50_order = z_values_df.groupby('Cluster')['benefit_score_scaled'].median().sort_values(ascending=False).index
# === Global Medians ===
median_ic50 = z_values_df['scaled_ic50'].median()
median_benefit = z_values_df['benefit_score_scaled'].median()
median_dependency = z_values_df['scaled_dep'].median()

# Set style
sns.set_style("white")

# Use Set2 color palette
palette = sns.color_palette("Set2", 3)  # Get 3 distinct colors
color_ic50 = palette[0]
color_dep = palette[2]
color_bs = palette[1]

# Create vertical subplots
fig, axes = plt.subplots(3, 1, figsize=(14, 18), sharex=False)
fig.suptitle('Distribution of Key Metrics Across Cell Line Clusters', fontsize=22, y=0.98)

# --- Plot 1: IC50 ---
sns.stripplot(data=z_values_df, x='Cluster', y='scaled_ic50', order=ic50_order,
              ax=axes[0], jitter=True, alpha=0.6, color=color_ic50)
sns.pointplot(data=z_values_df, x='Cluster', y='scaled_ic50', order=ic50_order,
              ax=axes[0], estimator=np.median, color="black",
              markers="_", scale=2.5, linestyles="", errorbar=None)
axes[0].axhline(median_ic50, color='black', linestyle='--', linewidth=1.5,
                label=f'Global Median ({median_ic50:.2f})')
axes[0].set_title('Erlotinib IC50', fontsize=18)
axes[0].set_ylabel('IC50 Value (mean-centered)', fontsize=14)
axes[0].legend()
sns.despine(ax=axes[0])

# --- Plot 2: Benefit Score ---
sns.stripplot(data=z_values_df, x='Cluster', y='benefit_score_scaled', order=ic50_order,
              ax=axes[1], jitter=True, alpha=0.6, color=color_bs)
sns.pointplot(data=z_values_df, x='Cluster', y='benefit_score_scaled', order=ic50_order,
              ax=axes[1], estimator=np.median, color="black",
              markers="_", scale=2.5, linestyles="", errorbar=None)
axes[1].axhline(median_benefit, color='black', linestyle='--', linewidth=1.5,
                label=f'Global Median ({median_benefit:.2f})')
axes[1].set_title('Benefit Score', fontsize=18)
axes[1].set_ylabel('Benefit Score (Min-Max scaled)', fontsize=14)
axes[1].legend()
sns.despine(ax=axes[1])

# --- Plot 3: EGFR Dependency ---
sns.stripplot(data=z_values_df, x='Cluster', y='scaled_dep', order=ic50_order,
              ax=axes[2], jitter=True, alpha=0.6, color=color_dep)
sns.pointplot(data=z_values_df, x='Cluster', y='scaled_dep', order=ic50_order,
              ax=axes[2], estimator=np.median, color="black",
              markers="_", scale=2.5, linestyles="", errorbar=None)
axes[2].axhline(median_dependency, color='black', linestyle='--', linewidth=1.5,
                label=f'Global Median ({median_dependency:.2f})')
axes[2].set_title('EGFR Dependency', fontsize=18)
axes[2].set_ylabel('Dependency Score (mean-centered)', fontsize=14)
axes[2].set_xlabel('Cluster ID', fontsize=14)
axes[2].legend()
sns.despine(ax=axes[2])

# Layout
plt.tight_layout(rect=[0, 0, 1, 0.96])

for coll in axes[0].collections:
    coll.set_rasterized(True)
for coll in axes[1].collections:
    coll.set_rasterized(True)
for coll in axes[2].collections:
    coll.set_rasterized(True)

fig.savefig("/home/sreeramp/cancer_dependency_project/nilabja/Approach3_Latent_factor/git_repo/Figs/clusterSpecific_scatterPlots_newHP_optuna.pdf", dpi=600, bbox_inches='tight')

plt.show()


In [ ]:
z_values_df['keyCluster'] = np.where(z_values_df['Cluster'].isin([13,24,15]), 'keyCluster', 'nonKey_cluster'),

In [ ]:
# Order
order = ['nonKey_cluster', 'key_cluster']

In [ ]:
from scipy.stats import mannwhitneyu
from statannotations.Annotator import Annotator

In [ ]:
# Extract groups
group1 = z_values_df[z_values_df['keyCluster'] == 'nonKey_cluster']['scaled_dep']
group2 = z_values_df[z_values_df['keyCluster'] == 'key_cluster']['scaled_dep']

# --- Mann–Whitney U test (two-sided) ---
U, pval = mannwhitneyu(group1, group2, alternative='two-sided')

# Sample sizes
n1, n2 = len(group1), len(group2)

# --- Effect size (rank-biserial correlation) ---
rbc = 1 - (2 * U) / (n1 * n2)

# --- Plot ---
plt.figure(figsize=(5, 4))

ax = sns.boxplot(
    data=z_values_df,
    x='keyCluster',
    y='scaled_dep',
    order=order,
    palette=['#D3D3D3', '#56B4E9'],  # grey + colorblind-friendly blue
    width=0.5,
    fliersize=0
)

sns.stripplot(
    data=z_values_df,
    x='keyCluster',
    y='scaled_dep',
    order=order,
    color='black',
    alpha=0.7,
    jitter=0.2,
    size=4
)

# --- Add statannotations (stars) ---
pairs = [('nonKey_cluster', 'key_cluster')]

annotator = Annotator(
    ax,
    pairs,
    data=z_values_df,
    x='keyCluster',
    y='scaled_dep',
    order=order
)

annotator.configure(
    test='Mann-Whitney',
    text_format='star',
    loc='outside',
    verbose=0
)

annotator.apply_and_annotate()

# --- Add detailed stats text ---
stats_text = (
    f"Mann–Whitney U = {U:.1f}\n"
    f"p = {pval:.2e}\n"
    f"n = ({n1}, {n2})\n"
    f"rank-biserial r = {rbc:.2f}"
)

plt.text(
    0.5, 1.15, stats_text,
    ha='center', va='center',
    transform=ax.transAxes,
    fontsize=10
)

# Labels
# plt.title('EGFR Dependency by Cluster Type', fontsize=14)
plt.xlabel('')
plt.ylabel('EGFR Dependency (mean-centered)', fontsize=12)

sns.despine()
plt.tight_layout()
# plt.savefig("/home/sreeramp/cancer_dependency_project/nilabja/Approach3_Latent_factor/git_repo/revision_final/EGFR_dep_keyCluster_boxplot.pdf", dpi=600, bbox_inches='tight')
plt.show()

In [ ]:
# Order
# order = ['Others', 'key_cluster']

# Extract groups
group1 = z_values_df[z_values_df['keyCluster'] == 'nonKey_cluster']['scaled_ic50']
group2 = z_values_df[z_values_df['keyCluster'] == 'key_cluster']['scaled_ic50']

# --- Mann–Whitney U test (two-sided) ---
U, pval = mannwhitneyu(group1, group2, alternative='two-sided')

# Sample sizes
n1, n2 = len(group1), len(group2)

# --- Effect size (rank-biserial correlation) ---
rbc = 1 - (2 * U) / (n1 * n2)

# --- Plot ---
plt.figure(figsize=(5, 4))

ax = sns.boxplot(
    data=z_values_df,
    x='keyCluster',
    y='scaled_ic50',
    order=order,
    palette=['#D3D3D3', '#56B4E9'],  # grey + colorblind-friendly blue
    width=0.5,
    fliersize=0
)

sns.stripplot(
    data=z_values_df,
    x='keyCluster',
    y='scaled_ic50',
    order=order,
    color='black',
    alpha=0.7,
    jitter=0.2,
    size=4
)

# --- Add statannotations (stars) ---
# pairs = [('Others', 'key_cluster')]

annotator = Annotator(
    ax,
    pairs,
    data=z_values_df,
    x='keyCluster',
    y='scaled_ic50',
    order=order
)

annotator.configure(
    test='Mann-Whitney',
    text_format='star',
    loc='outside',
    verbose=0
)

annotator.apply_and_annotate()

# --- Add detailed stats text ---
stats_text = (
    f"Mann–Whitney U = {U:.1f}\n"
    f"p = {pval:.2e}\n"
    f"n = ({n1}, {n2})\n"
    f"rank-biserial r = {rbc:.2f}"
)

plt.text(
    0.5, 1.15, stats_text,
    ha='center', va='center',
    transform=ax.transAxes,
    fontsize=10
)

# Labels
# plt.title('Erlotinib IC50 by Cluster Type', fontsize=14)
plt.xlabel('')
plt.ylabel('Erlotinib IC50 (mean-centered)', fontsize=12)

sns.despine()
plt.tight_layout()
# plt.savefig("/home/sreeramp/cancer_dependency_project/nilabja/Approach3_Latent_factor/git_repo/revision_final/Erlo_icr50_keyCluster_boxplot.pdf", dpi=600, bbox_inches='tight')
plt.show()

In [ ]:
z_values_df.head()

#### Model performances

In [ ]:
def plot_pred_vs_true(x, y, ax, title):
    # Stats
    pearson_r, _ = pearsonr(x, y)
    spearman_rho, _ = spearmanr(x, y)
    ci = concordance_index(x, y)

    # Regression fit
    a, b = np.polyfit(x, y, 1)
    y_fit = a * x + b
    r2 = r2_score(y, y_fit)

    # Plot scatter
    sns.scatterplot(x=x, y=y, s=40, ax=ax)

    # Regression line
    xx = np.linspace(x.min(), x.max(), 200)
    yy = a * xx + b
    ax.plot(xx, yy, linewidth=2, color='red')

    # Text box
    textstr = (
        f"Pearson r: {pearson_r:.3f}\n"
        f"Spearman ρ: {spearman_rho:.3f}\n"
        f"R² (fit): {r2:.3f}\n"
        f"CI: {ci:.3f}"
    )
    ax.text(
        0.05, 0.95, textstr,
        transform=ax.transAxes,
        fontsize=11, va='top',
        bbox=dict(boxstyle="round", fc="white", alpha=0.6)
    )

    ax.set_title(title)
    ax.set_xlabel("True")
    ax.set_ylabel("Predicted")

In [ ]:
train_exp = (G_train - forge_model.mean_exp.values) / forge_model.std_exp.values
test_exp  = (G_test  - forge_model.mean_exp.values) / forge_model.std_exp.values
# ---------------------------
# Create two plots
# ---------------------------
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# Dependency
x_dep_train = D_train.ravel()
y_dep_train = (train_exp @ forge_model.W @ forge_model.hD).ravel()

plot_pred_vs_true(x_dep_train, y_dep_train, axes[0,0],
                  "True vs Predicted Dependency (Train data)")

x_dep_test = D_test.ravel()
y_dep_test = (test_exp @ forge_model.W @ forge_model.hD).ravel()

plot_pred_vs_true(x_dep_test, y_dep_test, axes[0,1],
                  "True vs Predicted Dependency (Test data)")

# IC50
x_ic50_train = I_train.ravel()
y_ic50_train = (train_exp @ forge_model.W @ forge_model.hI).ravel()

plot_pred_vs_true(x_ic50_train, y_ic50_train, axes[1,0],
                  "True vs Predicted Dependency (Train data)")

x_ic50_test = I_test.ravel()
y_ic50_test = (test_exp @ forge_model.W @ forge_model.hI).ravel()

plot_pred_vs_true(x_ic50_test, y_ic50_test, axes[1,1],
                  "True vs Predicted IC50 (Test data)")

plt.tight_layout()
# plt.savefig("/home/sreeramp/cancer_dependency_project/nilabja/Approach3_Latent_factor/git_repo/Figs/EGFR_Erlotinib_newHP_performances_optuna.pdf", dpi=600, bbox_inches='tight')
plt.show()


In [ ]:
def compute_stats(x, y, alternative = 'two-sided'):
    U, p = mannwhitneyu(x, y, alternative=alternative)
    n1, n2 = len(x), len(y)
    rbc = 1 - (2 * U) / (n1 * n2)
    return U, p, rbc

In [ ]:
# benefit score quartile boxplots
# ---- 1. Create quartiles ----
z_values_df['quartile'] = pd.qcut(
    z_values_df['benefit_score_scaled'],
    q=4,
    labels=['Q1 (Low)', 'Q2', 'Q3', 'Q4 (High)']
)

order = ['Q1 (Low)', 'Q2', 'Q3', 'Q4 (High)']

# ---- 2. Color palette ----
palette = ['#9ecae1', "#b8adad", '#f4a582', '#ca0020']  # blue → grey → orange → red

# ---- 3. Plot function ----
def make_panel(ax, df, y, title_label):
    sns.boxplot(
        data=df,
        x='quartile',
        y=y,
        order=order,
        palette=palette,
        width=0.6,
        fliersize=0,
        ax=ax
    )

    sns.stripplot(
        data=df,
        x='quartile',
        y=y,
        order=order,
        color='black',
        alpha=0.4,
        jitter=0.2,
        size=3,
        ax=ax
    )

    # ---- styling ----
    ax.set_xlabel('')
    ax.set_title(title_label, fontsize=12)
    sns.despine(ax=ax)

# ---- 4. Split datasets ----
df_tr = z_values_df[z_values_df['set'] == 'train']
df_tst = z_values_df[z_values_df['set'] == 'test']

# ---- 5. Create 2x2 figure ----
fig, axes = plt.subplots(2, 2, figsize=(8, 6), sharex=True)

# Top row: EGFR Dependency
make_panel(axes[0, 0], df_tr, 'scaled_dep', 'Tr')
make_panel(axes[0, 1], df_tst, 'scaled_dep', 'Tst')

# Bottom row: IC50
make_panel(axes[1, 0], df_tr, 'scaled_ic50', 'Tr')
make_panel(axes[1, 1], df_tst, 'scaled_ic50', 'Tst')

# ---- 6. Axis labels ----
axes[0, 0].set_ylabel('EGFR Dep.')
axes[1, 0].set_ylabel('Erlotinib IC50')

axes[0, 1].set_ylabel('')
axes[1, 1].set_ylabel('')

# ---- 7. Global label ----
fig.text(0.5, 0.04, 'Benefit Score', ha='center', fontsize=12)

plt.tight_layout(rect=[0, 0.05, 1, 1])
# fig.savefig("/home/sreeramp/cancer_dependency_project/nilabja/Approach3_Latent_factor/git_repo/revision_final/forge_quartileBoxplots.pdf", dpi=600, bbox_inches='tight')
plt.show()

In [ ]:
U, p, rbc = compute_stats(df_tr[df_tr['quartile'] == 'Q1 (Low)']['scaled_dep'], df_tr[df_tr['quartile'] == 'Q2']['scaled_dep'], alternative='less')
print(f"Train EGFR Dependency: U={U:.1f}, p={p:.2e}, rbc={rbc:.2f}")
U, p, rbc = compute_stats(df_tr[df_tr['quartile'] == 'Q2']['scaled_dep'], df_tr[df_tr['quartile'] == 'Q3']['scaled_dep'], alternative='less')
print(f"Train EGFR Dependency: U={U:.1f}, p={p:.2e}, rbc={rbc:.2f}")
U, p, rbc = compute_stats(df_tr[df_tr['quartile'] == 'Q3']['scaled_dep'], df_tr[df_tr['quartile'] == 'Q4 (High)']['scaled_dep'], alternative='less')
print(f"Train EGFR Dependency: U={U:.1f}, p={p:.2e}, rbc={rbc:.2f}")

In [ ]:
U, p, rbc = compute_stats(df_tst[df_tst['quartile'] == 'Q1 (Low)']['scaled_dep'], df_tst[df_tst['quartile'] == 'Q2']['scaled_dep'], alternative='less')
print(f"Test EGFR Dependency: U={U:.1f}, p={p:.2e}, rbc={rbc:.2f}")
U, p, rbc = compute_stats(df_tst[df_tst['quartile'] == 'Q2']['scaled_dep'], df_tst[df_tst['quartile'] == 'Q3']['scaled_dep'], alternative='less')
print(f"Test EGFR Dependency: U={U:.1f}, p={p:.2e}, rbc={rbc:.2f}")
U, p, rbc = compute_stats(df_tst[df_tst['quartile'] == 'Q3']['scaled_dep'], df_tst[df_tst['quartile'] == 'Q4 (High)']['scaled_dep'], alternative='less')
print(f"Test EGFR Dependency: U={U:.1f}, p={p:.2e}, rbc={rbc:.2f}")

In [ ]:
U, p, rbc = compute_stats(df_tr[df_tr['quartile'] == 'Q1 (Low)']['scaled_ic50'], df_tr[df_tr['quartile'] == 'Q2']['scaled_ic50'], alternative='greater')
print(f"Train IC50: U={U:.1f}, p={p:.2e}, rbc={rbc:.2f}")
U, p, rbc = compute_stats(df_tr[df_tr['quartile'] == 'Q2']['scaled_ic50'], df_tr[df_tr['quartile'] == 'Q3']['scaled_ic50'], alternative='greater')
print(f"Train IC50: U={U:.1f}, p={p:.2e}, rbc={rbc:.2f}")
U, p, rbc = compute_stats(df_tr[df_tr['quartile'] == 'Q3']['scaled_ic50'], df_tr[df_tr['quartile'] == 'Q4 (High)']['scaled_ic50'], alternative='greater')
print(f"Train IC50: U={U:.1f}, p={p:.2e}, rbc={rbc:.2f}")

In [ ]:
U, p, rbc = compute_stats(df_tst[df_tst['quartile'] == 'Q1 (Low)']['scaled_ic50'], df_tst[df_tst['quartile'] == 'Q2']['scaled_ic50'], alternative='greater')
print(f"Test IC50: U={U:.1f}, p={p:.2e}, rbc={rbc:.2f}")
U, p, rbc = compute_stats(df_tst[df_tst['quartile'] == 'Q2']['scaled_ic50'], df_tst[df_tst['quartile'] == 'Q3']['scaled_ic50'], alternative='greater')
print(f"Test IC50: U={U:.1f}, p={p:.2e}, rbc={rbc:.2f}")
U, p, rbc = compute_stats(df_tst[df_tst['quartile'] == 'Q3']['scaled_ic50'], df_tst[df_tst['quartile'] == 'Q4 (High)']['scaled_ic50'], alternative='greater')
print(f"Test IC50: U={U:.1f}, p={p:.2e}, rbc={rbc:.2f}")

In [ ]:
z_values_df.to_csv('/home/sreeramp/cancer_dependency_project/forge_v2/FORGE/intermediate_data/egfr_erlotinib_fullRes_optuna.csv', index=True)